In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce_silver;

In [0]:
from pyspark.sql.functions import col, coalesce, lit

# Read from your bronze table
df_users_raw = spark.table("ecommerce.users_raw")

# Apply cleaning transformations
df_users_silver = (
    df_users_raw
    .dropDuplicates(["identifierHash"])
    .withColumn("language", coalesce(col("language"), lit("unknown")))
)

# Write as an external table explicitly mapped to your storage container
df_users_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .option("path", "abfss://landing@saecommercedataprod001.dfs.core.windows.net/silver/users/") \
    .saveAsTable("ecommerce_silver.users")

print("Successfully created silver.users table!")

Successfully created silver.users table!


In [0]:
# List of remaining tables to process into the silver layer
silver_tables = ["sellers", "buyers", "countries"]

for table in silver_tables:
    # Read from the respective bronze table
    df_raw = spark.table(f"ecommerce.{table}_raw")
    
    # Apply standard cleaning (e.g., dropping duplicates)
    df_silver = df_raw.dropDuplicates()
    
    # Write as an external Delta table in the silver layer path
    df_silver.write \
        .mode("overwrite") \
        .format("delta") \
        .option("path", f"abfss://landing@saecommercedataprod001.dfs.core.windows.net/silver/{table}/") \
        .saveAsTable(f"ecommerce_silver.{table}")
        
    print(f"Successfully created ecommerce_silver.{table} table!")

Successfully created ecommerce_silver.sellers table!
Successfully created ecommerce_silver.buyers table!
Successfully created ecommerce_silver.countries table!
